# TAE-IA · Módulo 6 · L23 — Generating Music and Sound: MusicGen and AudioLDM2

| | |
|---|---|
| **Models** | MusicGen-small (transformer over EnCodec tokens, 2.4 GB) · AudioLDM2 (latent diffusion, ~4.5 GB) |
| **Also used** | CLAP — AudioLDM2's own text/audio encoder — to score prompt adherence |
| **Runtime** | T4 GPU · ~7 GB on the runtime disk, re-downloaded each session |
| **Outputs** | `TAE_IA_M6/L23_output/` on Drive |
| **Licences** | MusicGen weights CC BY-NC 4.0 · AudioLDM2 weights CC BY-NC-SA 4.0 — **non-commercial** |

## Learning objectives

1. Compute what a text-to-music transformer actually generates: tokens per second, codebooks, bits
2. Measure how guidance and temperature trade prompt adherence against quality and variety
3. Generate sound effects with latent diffusion and measure what the number of steps buys
4. Score prompt adherence with CLAP — and find where the score disagrees with your ears
5. Mix generated music and effects into one scene with correct resampling and gain

## 1 — Setup

`transformers` is already on Colab; `diffusers` needs an update. Weights go to the runtime disk.

In [ ]:
!pip install -q -U diffusers accelerate

In [ ]:
import os, time, random
import numpy as np
import torch

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/TAE_IA_M6'
OUTPUT_DIR = f'{DRIVE_ROOT}/L23_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_CACHE = '/content/models'
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['HF_HOME'] = MODEL_CACHE

if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

import soundfile as sf, librosa, librosa.display
import matplotlib.pyplot as plt
import IPython.display as ipd
import pandas as pd
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

def save(wav, sr, name):
    path = f'{OUTPUT_DIR}/{name}.wav'
    sf.write(path, np.asarray(wav, dtype=np.float32), sr)
    return path

def play(wav, sr, label=''):
    print(f'{label}  ({len(wav) / sr:.1f} s)')
    ipd.display(ipd.Audio(wav, rate=sr))

def spec_grid(clips, titles, path, ncols=3, fmax=8000):
    n = len(clips); nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 2.8 * nrows), squeeze=False)
    for ax, (wav, sr), title in zip(axes.flat, clips, titles):
        S_db = librosa.power_to_db(librosa.feature.melspectrogram(y=wav, sr=sr, n_mels=96, fmax=fmax), ref=np.max)
        librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='mel', fmax=fmax, ax=ax)
        ax.set_title(title, fontsize=9)
    for ax in list(axes.flat)[n:]:
        ax.axis('off')
    plt.tight_layout(); plt.savefig(path); plt.show()

def vram(tag=''):
    print(f'  VRAM {torch.cuda.memory_allocated() / 1e9:5.2f} GB   {tag}')

print(torch.cuda.get_device_name(0))

## 2 — MusicGen: what it actually generates

MusicGen never produces samples. It produces **EnCodec tokens**: several codebooks per frame, at a
fixed frame rate. The config tells you the whole budget before you generate anything.

In [ ]:
from transformers import AutoProcessor, MusicgenForConditionalGeneration

t0 = time.time()
mg_proc  = AutoProcessor.from_pretrained('facebook/musicgen-small')
musicgen = MusicgenForConditionalGeneration.from_pretrained('facebook/musicgen-small', dtype=torch.float16).to('cuda').eval()
print(f'MusicGen-small loaded in {time.time() - t0:.0f} s'); vram('MusicGen')

MG_SR         = musicgen.config.audio_encoder.sampling_rate
FRAME_RATE    = musicgen.config.audio_encoder.frame_rate
N_CODEBOOKS   = musicgen.config.decoder.num_codebooks
CODEBOOK_SIZE = musicgen.config.audio_encoder.codebook_size

seconds = 10
tokens = seconds * FRAME_RATE * N_CODEBOOKS
bitrate = FRAME_RATE * N_CODEBOOKS * np.log2(CODEBOOK_SIZE)
pcm = MG_SR * 16
print(f'sample rate {MG_SR} Hz | frame rate {FRAME_RATE} Hz | {N_CODEBOOKS} codebooks x {CODEBOOK_SIZE} codes')
print(f'{seconds} s of music = {seconds * FRAME_RATE} frames = {tokens} tokens')
print(f'token bitrate {bitrate:.0f} bit/s vs 16-bit PCM {pcm} bit/s -> {pcm / bitrate:.0f}x smaller')

In [ ]:
def musicgen_generate(prompts, seconds=10, guidance=3.0, temperature=1.0, seed=SEED):
    torch.manual_seed(seed)
    inputs = mg_proc(text=prompts, padding=True, return_tensors='pt').to('cuda')
    torch.cuda.synchronize(); t0 = time.time()
    with torch.inference_mode():
        out = musicgen.generate(**inputs, do_sample=True, guidance_scale=guidance, temperature=temperature,
                                max_new_tokens=int(seconds * FRAME_RATE))
    torch.cuda.synchronize()
    return [out[i, 0].float().cpu().numpy() for i in range(len(prompts))], time.time() - t0

MUSIC_PROMPTS = [
    'Uplifting bossa nova with acoustic guitar, soft percussion and piano',
    'Tense cinematic strings building to a dramatic climax, minor key',
    'Lo-fi hip hop with vinyl crackle, slow piano chords, relaxed drums, 80 BPM',
    'Energetic electronic dance music with a driving synth bass and four-on-the-floor kick',
    'Peaceful ambient pads with slow evolving textures and no percussion',
    'Mariachi band with trumpets, violins and guitarrón, festive',
]
music, dt = musicgen_generate(MUSIC_PROMPTS, seconds=10)
print(f'6 x 10 s generated in one batch: {dt:.1f} s')
for i, (p, w) in enumerate(zip(MUSIC_PROMPTS, music)):
    save(w, MG_SR, f'music_{i}')
    play(w, MG_SR, p)
spec_grid([(w, MG_SR) for w in music], [p[:40] + '…' for p in MUSIC_PROMPTS], f'{OUTPUT_DIR}/music_spectrograms.png')

**Observation.** Which prompts did it follow, and which elements did it ignore (the BPM? *no
percussion*? the mariachi instruments)? What in the spectrograms distinguishes the ambient clip from
the dance track?

*Your answer:*

## 3 — Guidance and temperature

**Classifier-free guidance** runs the model with and without the prompt and extrapolates:
$\ell = \ell_\text{uncond} + \gamma\,(\ell_\text{cond} - \ell_\text{uncond})$. `guidance=1` switches it off.
**Temperature** flattens or sharpens the token distribution. Same prompt, same seed — only one knob moves.

In [ ]:
BASE = 'Uplifting bossa nova with acoustic guitar, soft percussion and piano'
sweep = {}
for g in (1.0, 3.0, 6.0, 10.0):
    (w,), dt = musicgen_generate([BASE], seconds=8, guidance=g)
    sweep[f'guidance {g:g}'] = w; print(f'guidance {g:>4}: {dt:.1f} s')
for temp in (0.5, 1.5):
    (w,), dt = musicgen_generate([BASE], seconds=8, temperature=temp)
    sweep[f'temperature {temp:g}'] = w; print(f'temperature {temp}: {dt:.1f} s')
for k, w in sweep.items():
    save(w, MG_SR, 'sweep_' + k.replace(' ', '_')); play(w, MG_SR, k)
spec_grid([(w, MG_SR) for w in sweep.values()], list(sweep), f'{OUTPUT_DIR}/guidance_temperature.png')

**Observation.** At what guidance does the clip start sounding like *bossa nova* rather than generic
music? At what value does it start sounding worse? Did `guidance 1` cost less time than `guidance 3`?
What did temperature 0.5 and 1.5 do?

*Your answer:*

In [ ]:
del musicgen; torch.cuda.empty_cache(); vram('MusicGen freed')

## 4 — AudioLDM2: sound effects by latent diffusion

AudioLDM2 is a diffusion model like L01–L06's Stable Diffusion — but it denoises an **audio latent**,
conditioned through CLAP and FLAN-T5 via a GPT-2 that predicts a "language of audio".

Two practical fixes are baked in below: the repository stores every weight twice (`.bin` and
`.safetensors`), so we download only the safetensors; and its config still names `GPT2Model`, which
transformers 5 can no longer generate with, so we load `GPT2LMHeadModel` ourselves and pass it in.

In [ ]:
from huggingface_hub import snapshot_download
from diffusers import AudioLDM2Pipeline
from transformers import GPT2LMHeadModel

t0 = time.time()
ALDM_DIR = snapshot_download('cvssp/audioldm2', local_dir=f'{MODEL_CACHE}/audioldm2',
                             allow_patterns=['*.json', '*.txt', '*.model', '*.safetensors'])
lm = GPT2LMHeadModel.from_pretrained(f'{ALDM_DIR}/language_model', dtype=torch.float16)
aldm = AudioLDM2Pipeline.from_pretrained(ALDM_DIR, language_model=lm, dtype=torch.float16).to('cuda')
aldm.set_progress_bar_config(disable=True)
ALDM_SR = aldm.vocoder.config.sampling_rate
print(f'AudioLDM2 loaded in {time.time() - t0:.0f} s, output {ALDM_SR} Hz'); vram('AudioLDM2')

def aldm_generate(prompts, seconds=5.0, steps=50, guidance=3.5, n_waveforms=1, seed=SEED, negative='Low quality.'):
    prompts = [prompts] if isinstance(prompts, str) else prompts
    g = torch.Generator('cuda').manual_seed(seed)
    torch.cuda.synchronize(); t0 = time.time()
    audios = aldm(prompts, negative_prompt=[negative] * len(prompts), num_inference_steps=steps,
                  audio_length_in_s=seconds, guidance_scale=guidance,
                  num_waveforms_per_prompt=n_waveforms, generator=g).audios
    torch.cuda.synchronize()
    return [np.asarray(a, dtype=np.float32) for a in audios], time.time() - t0

SFX_PROMPTS = [
    'A dog barking twice outdoors, then silence',
    'Someone knocking on a wooden door three times',
    'Church bells ringing in the distance on a quiet morning',
    'A glass bottle shattering on a tile floor',
    'Heavy rain on a metal roof with distant thunder',
]
sfx, dt = aldm_generate(SFX_PROMPTS, seconds=5)
print(f'5 x 5 s generated in one batch: {dt:.1f} s')
for i, (p, w) in enumerate(zip(SFX_PROMPTS, sfx)):
    save(w, ALDM_SR, f'sfx_{i}'); play(w, ALDM_SR, p)
spec_grid([(w, ALDM_SR) for w in sfx], [p[:40] + '…' for p in SFX_PROMPTS], f'{OUTPUT_DIR}/sfx_spectrograms.png')

**Observation.** Did it count — two barks, three knocks? Which prompt produced a sound you could
*recognise* without reading the prompt, and which produced something generic?

*Your answer:*

## 5 — What diffusion steps buy

MusicGen's cost grows with **duration** (one step per frame). A diffusion model's cost grows with the
number of **denoising steps**, whatever the duration. Measure both sides of that claim.

In [ ]:
PROMPT = 'Church bells ringing in the distance on a quiet morning'
steps_rows, steps_clips = [], []
for steps in (10, 25, 50, 100):
    (w,), dt = aldm_generate(PROMPT, seconds=5, steps=steps)
    steps_rows.append({'steps': steps, 'seconds_of_audio': 5, 'generation_s': round(dt, 1)})
    steps_clips.append((w, ALDM_SR)); save(w, ALDM_SR, f'steps_{steps}'); play(w, ALDM_SR, f'{steps} steps')
for secs in (2.5, 10.0):
    (w,), dt = aldm_generate(PROMPT, seconds=secs, steps=50)
    steps_rows.append({'steps': 50, 'seconds_of_audio': secs, 'generation_s': round(dt, 1)})
display(pd.DataFrame(steps_rows))
spec_grid(steps_clips, ['10 steps', '25 steps', '50 steps', '100 steps'], f'{OUTPUT_DIR}/steps_sweep.png', ncols=4)

**Observation.** Does generation time scale with steps, with duration, or both? Where did quality stop
improving? Compare with MusicGen's timings from Section 3.

*Your answer:*

## 6 — Scoring prompt adherence with CLAP

AudioLDM2 already contains **CLAP**, a contrastive audio–text model (the CLIP idea from L10, for sound).
Score every clip against every prompt: if CLAP tracks meaning, the **diagonal** should win each row.

In [ ]:
clap, clap_fe, clap_tok = aldm.text_encoder, aldm.feature_extractor, aldm.tokenizer
CLAP_SR = clap_fe.sampling_rate

def _embedding(out):
    return out.pooler_output if hasattr(out, 'pooler_output') else out

@torch.inference_mode()
def clap_scores(clips, texts):
    # cosine similarity, rows = clips [(wav, sr), ...], columns = texts
    wavs = [librosa.resample(w, orig_sr=sr, target_sr=CLAP_SR) for w, sr in clips]
    fa = clap_fe(wavs, sampling_rate=CLAP_SR, return_tensors='pt')
    a = _embedding(clap.get_audio_features(input_features=fa['input_features'].to('cuda', torch.float16),
                                           is_longer=fa['is_longer'].to('cuda')))
    tt = clap_tok(texts, padding=True, return_tensors='pt').to('cuda')
    t = _embedding(clap.get_text_features(**tt))
    a = torch.nn.functional.normalize(a.float(), dim=-1)
    t = torch.nn.functional.normalize(t.float(), dim=-1)
    return (a @ t.T).cpu().numpy()

clips  = [(w, MG_SR) for w in music] + [(w, ALDM_SR) for w in sfx]
prompts = MUSIC_PROMPTS + SFX_PROMPTS
M = clap_scores(clips, prompts)
labels = [f'M{i}' for i in range(len(music))] + [f'S{i}' for i in range(len(sfx))]
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(M, cmap='viridis'); plt.colorbar(im, ax=ax, label='CLAP cosine similarity')
ax.set_xticks(range(len(labels)), labels); ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel('prompt'); ax.set_ylabel('generated clip'); ax.set_title('Does each clip match its own prompt best?')
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/clap_matrix.png'); plt.show()
hits = [int(np.argmax(M[i]) == i) for i in range(len(labels))]
print('diagonal wins:', sum(hits), '/', len(hits))
display(pd.DataFrame({'clip': labels, 'own prompt score': np.diag(M).round(3), 'best prompt': [labels[j] for j in M.argmax(1)]}))

**Observation.** How many clips matched their own prompt best? For a miss, listen: was CLAP wrong, or
was the generation wrong? Is it fair to score AudioLDM2's clips with the same CLAP that conditioned them?

*Your answer:*

## 7 — Mix a scene

A music bed plus effects placed in time. Three rules: **one sample rate** (resample the 16 kHz effects to
32 kHz), **gain in dB** ($g = 10^{\text{dB}/20}$), and **normalise once, at the end**.

In [ ]:
def to_rate(w, sr, target=MG_SR):
    return w if sr == target else librosa.resample(w, orig_sr=sr, target_sr=target)

def fade(w, sr, ms=20):
    n = min(int(sr * ms / 1000), len(w) // 2)
    ramp = np.linspace(0, 1, n, dtype=np.float32)
    w = w.copy(); w[:n] *= ramp; w[-n:] *= ramp[::-1]
    return w

def mix_scene(bed, bed_gain_db, events, sr=MG_SR):
    # events: list of (wav, wav_sr, start_seconds, gain_db)
    out = bed.astype(np.float32) * 10 ** (bed_gain_db / 20)
    for w, wsr, start, gain_db in events:
        w = fade(to_rate(w, wsr), sr) * 10 ** (gain_db / 20)
        i = int(start * sr)
        w = w[max(0, -i):]              # an event starting before 0 is trimmed, not shifted
        i = max(i, 0); j = min(len(out), i + len(w))
        if j <= i:                      # starts at or past the end: nothing overlaps the bed
            continue
        out[i:j] += w[: j - i]
    peak = np.abs(out).max()
    return 0.9 * out / peak if peak > 0 else out

scene = mix_scene(music[4], -6, [(sfx[4], ALDM_SR, 0.0, -3), (sfx[2], ALDM_SR, 4.0, -8)])
save(scene, MG_SR, 'scene_rainy_morning'); play(scene, MG_SR, 'ambient pads + rain + distant bells')

## Exercise 1 — Does a more specific prompt score higher?

Pick **one idea** (a sound effect or a piece of music). Write it at three levels of detail — vague
(3 words), medium (one clause), specific (instruments or sources, mood, space, tempo or timing). Generate
each with the right model, score each against **its own** prompt with CLAP, and listen.

Does more detail raise the CLAP score? Does it make the audio better — and are those the same thing?

In [ ]:
# Exercise 1 -- Prompt specificity
# Given: aldm_generate(prompts, seconds=...), clap_scores(clips, texts), save, play, ALDM_SR
# (MusicGen was freed in Section 3 -- use AudioLDM2 here, it generates music too.)

LEVELS = [
    # TODO 1: 'vague prompt', 'medium prompt', 'specific prompt'  -- the same idea three times
]

# TODO 2: generate each prompt (5 s, same seed) and save/play them
# TODO 3: score each clip against its own prompt: clap_scores([(w, ALDM_SR)], [prompt])[0, 0]
# TODO 4: show a small table level / prompt / score, then answer below

**Exercise 1 — Answer:**
*Your answer:*

## Exercise 2 — Assemble a 10-second scene

Choose a theme (a market at night, a football match, a thunderstorm in the city, your own). Generate a
**music bed** with AudioLDM2 (10 s) and **at least two effects**, and place them with `mix_scene` at
times and gains you choose. Save it as `ex2_scene.wav` and explain three mixing decisions.

In [ ]:
# Exercise 2 -- Scene assembly
# Given: aldm_generate, mix_scene(bed, bed_gain_db, events), save, play, ALDM_SR, MG_SR

THEME = 'TODO: your theme'

# TODO 1: generate a 10 s music or ambience bed with aldm_generate(...)
# TODO 2: generate at least two effects (3-5 s each)
# TODO 3: resample the bed to MG_SR (librosa.resample) and call mix_scene with your times and gains
# TODO 4: save(scene, MG_SR, 'ex2_scene') and play it

**Exercise 2 — Answer:** *theme, then three decisions (what, when, how loud, and why).*
*Your answer:*

## Critical Analysis

### Q1 — The guidance trade-off

From your Section 3 sweep: what does guidance buy and what does it cost? Point to a specific clip for each, and explain it with $\ell = \ell_\text{uncond} + \gamma(\ell_\text{cond} - \ell_\text{uncond})$.

*Your answer:*

### Q2 — Tokens or steps: where does the time go?

Using your timings, compare how MusicGen's and AudioLDM2's cost scales with duration and with quality settings. Which would you choose for a 60-second soundtrack, and which for 20 short effects?

*Your answer:*

### Q3 — Evaluating generated audio

Where did your CLAP matrix disagree with your ears? Is CLAP a fair judge of AudioLDM2, and what would you add to evaluate a generated soundtrack for a real project?

*Your answer:*

### Q4 — Whose music is this?

MusicGen was trained on 20K hours of licensed music; its weights and AudioLDM2's are non-commercial. Could you use today's outputs in your final project's demo? In a product? What would you disclose, and to whom?

*Your answer:*

## Submission Checklist

- [ ] `music_spectrograms.png`, `guidance_temperature.png`, `sfx_spectrograms.png`, `steps_sweep.png`, `clap_matrix.png` in `L23_output/`
- [ ] `scene_rainy_morning.wav` and `ex2_scene.wav` saved
- [ ] Observation cells in Sections 2–6 answered
- [ ] Exercises 1 and 2 with their answers
- [ ] Critical Analysis Q1–Q4, citing **your** timings, scores and clips

## Before You Close This Tab

Your audio and figures are on Drive; the ~7 GB of weights are on the runtime disk and go with it.
**Runtime → Disconnect and delete runtime.**